In [ ]:
import os 

os.chdir("..")

In [ ]:
from jp_imports import JPTrade
from datetime import datetime
import polars as pl

jt = JPTrade()

In [ ]:
df = jt.process_int_jp(
            time_frame="qtr",
            level="hts",
            agriculture_filter=True,
            source="org",
            corrections=True,
        )
df = df.with_columns(
    hs4=pl.col("hts_code").str.slice(0, 4),
    imports_qty=pl.when(pl.col("imports_qty") == 0)
    .then(1)
    .otherwise(pl.col("imports_qty")),
    exports_qty=pl.when(pl.col("exports_qty") == 0)
    .then(1)
    .otherwise(pl.col("exports_qty")),
)
df = df.group_by(pl.col("year", "qtr", "hs4")).agg(
    imports=pl.col("imports").sum(),
    exports=pl.col("exports").sum(),
    imports_qty=pl.col("imports_qty").sum(),
    exports_qty=pl.col("exports_qty").sum(),
)
df = df.with_columns(
    price_imports=pl.col("imports") / pl.col("imports_qty"),
    price_exports=pl.col("exports") / pl.col("exports_qty"),
    date=pl.datetime(
            pl.col("year"), 
            (pl.col("qtr") - 1) * 3 + 1, 
            1
        ),
).sort(["hs4", "date"]).with_columns(
        # Calculate Year-over-Year (YoY) growth (lag of 4 quarters)
        price_imports_yoy=pl.col("price_imports").pct_change(4).over("hs4") * 100,
        price_exports_yoy=pl.col("price_exports").pct_change(4).over("hs4") * 100,
    )
df

In [ ]:


# 2. Set up the plotting environment
plt.figure(figsize=(14, 7))
sns.set_theme(style='whitegrid')

# 3. Create the line plot grouped by 'hs4'
# Note: If you have too many unique 'hs4' codes, consider filtering for top categories first.
sns.lineplot(
    data=df.filter(pl.col("year") >= 2025),
    x='date',
    y='price_imports',
    hue='hs4',
    alpha=0.8,
)

# 4. Customize labels and layout
plt.title(
    'Year-over-Year Percentage Change of Imports by HS4 Code',
    fontsize=14,
    pad=15,
)
plt.xlabel('Date', fontsize=12)
plt.ylabel('YoY Import % Change', fontsize=12)

# Place the legend outside the plot area if there are multiple categories
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    title='HS4 Code',
    fontsize='small',
)
plt.tight_layout()

# 5. Display the plot
plt.show()

In [ ]:
import plotly.graph_objects as go
import polars as pl

# 1. Filter for the latest date. Keep actual values for sorting/labels, 
# and create a separate column clipped to +/- 100 for the bar lengths.
latest_data = (
    df.filter(pl.col("date") == pl.col("date").max())
    .with_columns(
        actual_yoy=pl.col("price_imports_yoy"),
        plot_yoy=pl.col("price_imports_yoy").clip(-100, 100)
    )
)

# 2. Get Top 10 and Bottom 10 based on actual (unclipped) values
top_10_data = (
    latest_data.sort("actual_yoy", descending=True).head(10).to_pandas()
)
bottom_10_data = (
    latest_data.sort("actual_yoy", descending=False).head(10).to_pandas()
)

# 3. Create base figure
fig = go.Figure()

# 4. Add Horizontal Bar Chart for Top 10 (Default visible) 
# Bar length uses 'plot_yoy' (capped at 100), but text displays the 'actual_yoy' value
fig.add_trace(
    go.Bar(
        x=top_10_data["plot_yoy"],
        y=top_10_data["hs4"],
        orientation="h",
        text=top_10_data["actual_yoy"].round(1).astype(str) + "%",
        textposition="auto",
        marker=dict(
            color="rgba(31, 119, 180, 0.6)",
            line=dict(color="rgb(31, 119, 180)", width=1),
        ),
        showlegend=False,
        visible=True,
    )
)

# 5. Add Horizontal Bar Chart for Bottom 10 (Initially hidden)
fig.add_trace(
    go.Bar(
        x=bottom_10_data["plot_yoy"],
        y=bottom_10_data["hs4"],
        orientation="h",
        text=bottom_10_data["actual_yoy"].round(1).astype(str) + "%",
        textposition="auto",
        marker=dict(
            color="rgba(255, 127, 14, 0.6)",
            line=dict(color="rgb(255, 127, 14)", width=1),
        ),
        showlegend=False,
        visible=False,
    )
)

# 6. Layout adjustments with tab-style toggle buttons
fig.update_layout(
    template="plotly_white",
    title_text="Price Imports YoY % Change: Top 10 HS4 Categories",
    height=600,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=0,  # Default to Top 10
            x=0.5,
            y=1.15,
            xanchor="center",
            yanchor="top",
            buttons=[
                dict(
                    label="Top 10",
                    method="update",
                    args=[
                        {"visible": [True, False]},
                        {
                            "title": "Price Imports YoY % Change: Top 10 HS4 Categories"
                        },
                    ],
                ),
                dict(
                    label="Bottom 10",
                    method="update",
                    args=[
                        {"visible": [False, True]},
                        {
                            "title": "Price Imports YoY % Change: Bottom 10 HS4 Categories"
                        },
                    ],
                ),
            ],
        )
    ],
)

# Set fixed X-axis range to cleanly cap at +/- 100% (+/- 105 for padding)
fig.update_xaxes(title_text="Price Imports YoY (% Change)", range=[-105, 105])

fig.add_vline(x=0, line_dash="dash", line_color="red", opacity=0.7)

fig.show()